# Chapter 14: Retrieval, Agents, and Multimodal Models

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch14_retrieval_agents_and_multimodal_models.ipynb)


## What is in this notebook, and what to change in it

Three cells, one per topic the chapter covers, and none of them calls a model.
Each replaces the model with a stub so that the machinery around it is visible.

1. **A minimal RAG pipeline**: index, retrieve by cosine similarity, generate.
   The embedding function is a stand-in that seeds a random generator from the
   text, and the generator is a lambda returning a fixed string.
2. **The ReAct loop**, thought then action then observation, with the tool
   dispatch and the regular expressions that parse the model's output. Defined
   and not called: the usage is two commented lines at the bottom, because
   calling it needs both a model and tools.
3. **A LoRA linear layer**, with frozen base weights and two low-rank matrices,
   and a parameter count for d = 4096 at rank 16. The cell's comment predicts
   16,908,288 parameters total and 131,072 trainable, at 0.78 percent. Check it.

**Look at what cell 1 actually retrieves.** Asked "What is the capital of
France?", it returns "The Eiffel Tower is 330m tall." and "Python was created in
1991.", and leaves "Paris is the capital of France." sitting in the index. That
is correct behaviour for this code and the most useful thing in the cell. The
embedding is a random vector keyed on the text, so cosine similarity between a
question and a document measures nothing whatever, and the top-k it returns is
an arbitrary ranking of noise. Every part of the pipeline works and the
retrieval is worthless.

That is what a RAG system with a bad encoder looks like from the outside, and it
is the failure mode hardest to notice in a real one: no error, no exception, a
fluent answer, and the wrong documents underneath it. The pipeline shape is what
this cell teaches. Swap `fake_embed` for a sentence encoder and the same code
starts answering.

**One further property of cell 1, related and separate.** The stand-in embedding
seeds itself from Python's `hash` of the text, and Python
randomises string hashing per process unless `PYTHONHASHSEED` is set before the
interpreter starts. So the vectors, and with them the ranking the retriever
returns, differ between runs in different processes, even though the comment
above them says the mapping is deterministic. It is deterministic within one
run. The stored output below was produced with `PYTHONHASHSEED=0`, and running
it yourself without that will very likely retrieve different documents.

Replacing `hash(text)` with `int.from_bytes(hashlib.sha256(text.encode()).digest()[:4], "big")`
fixes it, and doing that is a better five minutes than anything else in this
file: a retrieval system whose index depends on which process built it is a bug
with a real cost, and this is the smallest version of it.

For cell 3, sweep the rank. Try 1, 4, 16 and 64, printing the trainable fraction
each time. The whole case for LoRA is the shape of that curve.


> **This notebook was executed when it was built**, so the output under each
> cell is a real run's and you can read the file without running anything.
> Rebuild it with `python tools/build_notebook.py ch14`.


In [1]:
# Seeded before the chapter's own cells run.
#
# The cells below draw on torch and numpy. This notebook is committed with its
# output stored, and a stored number that changes on every rebuild is
# noise printed as a result. The bundle originals are read only and
# cannot be fixed where they live, so they are seeded here instead.
#
# Same seed as tools/claim_instances.py, which produced the slide
# numbers, so a number that appears in both places appears once.
import torch
import numpy as np
torch.manual_seed(20260729)
np.random.seed(20260729)
print('seeded torch, numpy with 20260729')

seeded torch, numpy with 20260729


### 14.1.2 The RAG Pipeline: Index, Retrieve, Generate

![Figure 14.1 -- RAG Pipeline Architecture](../figures/fig-14-1.pdf)


In [2]:
import numpy as np

class SimpleRAG:
    """Minimal RAG: embed documents, retrieve by cosine sim, generate."""
    def __init__(self, docs, embed_fn, generate_fn, k=3):
        self.docs = docs
        self.generate_fn = generate_fn
        self.k = k
        # Index: embed all documents
        self.doc_embeddings = np.array([embed_fn(d) for d in docs])

    def retrieve(self, query_emb):
        """Find top-k documents by cosine similarity."""
        sims = self.doc_embeddings @ query_emb
        norms = np.linalg.norm(self.doc_embeddings, axis=1)
        sims /= (norms * np.linalg.norm(query_emb) + 1e-8)
        top_k = np.argsort(sims)[-self.k:][::-1]
        return [(self.docs[i], float(sims[i])) for i in top_k]

    def query(self, question, embed_fn):
        q_emb = embed_fn(question)
        retrieved = self.retrieve(q_emb)
        context = "\n".join([f"[Doc]: {doc}" for doc, _ in retrieved])
        prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
        return self.generate_fn(prompt), retrieved

# Demo with content-seeded random embeddings so that the same text
# deterministically maps to the same vector (a true embedding would
# do this too). In production, replace with a sentence-transformer.
def fake_embed(text):
    rng = np.random.RandomState(hash(text) & 0xffffffff)
    return rng.randn(64)
fake_generate = lambda prompt: "Generated answer based on context."
docs = ["Paris is the capital of France.", "Berlin is in Germany.",
        "The Eiffel Tower is 330m tall.", "Python was created in 1991."]
rag = SimpleRAG(docs, fake_embed, fake_generate, k=2)
answer, sources = rag.query("What is the capital of France?", fake_embed)
print(f"Answer: {answer}")
print(f"Sources: {[doc for doc, score in sources]}")


Answer: Generated answer based on context.
Sources: ['The Eiffel Tower is 330m tall.', 'Python was created in 1991.']


### 14.2.1 The ReAct Framework: Thought-Action-Observation

![Figure 14.2 -- The ReAct Agent Loop](../figures/fig-14-2.pdf)


In [3]:
import re

def react_agent(question, tools, llm, max_steps=5):
    """Simple ReAct loop: Thought -> Action -> Observation."""
    context = f"Question: {question}\n"
    for step in range(max_steps):
        # LLM generates Thought + (Action or Answer)
        response = llm(context + "Thought:")
        context += f"Thought: {response}\n"
        # Check for final answer
        answer_match = re.search(r"Answer:\s*(.+)", response)
        if answer_match:
            return answer_match.group(1).strip()
        # Parse action: tool_name[tool_input]
        action_match = re.search(r"Action:\s*(\w+)\[(.+?)\]", response)
        if not action_match:
            return response  # fallback: return raw response
        tool_name, tool_input = action_match.groups()
        if tool_name in tools:
            observation = tools[tool_name](tool_input)
        else:
            observation = f"Error: unknown tool '{tool_name}'"
        context += f"Action: {tool_name}[{tool_input}]\n"
        context += f"Observation: {observation}\n"
    return "Max steps reached without final answer."

# Usage: tools = {"search": search_fn, "calc": calculator_fn}
# answer = react_agent("What is 2 * the population of Paris?", tools, llm)
# print(f"Final answer: {answer}")


### 14.5.2 LoRA and Parameter-Efficient Fine-Tuning

The following code implements a LoRA-augmented linear layer and demonstrates the parameter savings.


In [4]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA: W' = W_0 + B @ A (W_0 frozen)."""
    def __init__(self, in_features, out_features, rank=4):
        super().__init__()
        self.W0 = nn.Linear(in_features, out_features, bias=False)
        self.W0.weight.requires_grad = False  # freeze pretrained
        # LoRA matrices: B (out x rank), A (rank x in)
        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x):
        base = self.W0(x)                # frozen forward pass
        lora = x @ self.A.T @ self.B.T   # low-rank update
        return base + lora

    def merge_weights(self):
        """Merge LoRA into base weights (zero overhead at inference)."""
        self.W0.weight.data += self.B @ self.A

# Compare parameter counts for d=4096, rank=16
d_in, d_out, rank = 4096, 4096, 16
layer = LoRALinear(d_in, d_out, rank)
total = sum(p.numel() for p in layer.parameters())
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print(f"Total params: {total:,}")
print(f"Trainable:    {trainable:,} ({trainable/total:.2%})")
# Total params: 16,908,288
# Trainable:    131,072 (0.78%)


Total params: 16,908,288
Trainable:    131,072 (0.78%)


---

## Summary

This notebook demonstrated the key code examples from Chapter 14: Retrieval, Agents, and Multimodal Models. For the full mathematical exposition and discussion, refer to the textbook chapter.
